# Text Span JEPA v3 — SIGReg без EMA, anti-collapse

**Цель:** исправить обнаруженный коллапс модели v2. v2 на 20M токенов дала target_std≈0.00113, retrieval@1≈1/8 и SIGReg≈0.482847, практически равный значению для нулевого вектора в ошибочной регуляризации.

Этот ноутбук — **исследовательская адаптация**, не точная реплика опубликованной LeJEPA для изображений. Мы сохраняем 12×768 Transformer на FineWeb-Edu и ставим предсказание **каждого скрытого токенового латента** с контролем SIGReg по ~8×128=1024 латентным векторам вместо 8 pooled-векторов.

Что изменилось:
- **Нет EMA и stop-gradient**: контекст и скрытый блок обрабатывает один общий энкодер; оба пути дифференцируемы.
- **Sliced Epps–Pulley SIGReg**, основанный на эталонной формуле LeJEPA: интегрирование с Gaussian window и множителем числа наблюдений N.
- Прогноз на **каждую из 128 скрытых позиций**, а не на среднее скрытого блока.
- SIGReg на тысячах *токеновых представлений* за microbatch. Эти представления **коррелированы**, поэтому эффективная статистическая выборка меньше 1024 — это осознанный компромисс.
- Отдельная **same-position retrieval@1** и сравнение с перемешанным текстом, проверка полного коллапса, автоматический checkpoint, сохранение метрик и публикация в GitHub.
- Профиль **base**: 12×768 (~125M), **large**: 16×1024 (~250M). Данные — FineWeb-Edu streaming, токенизатор GPT-2 (веса GPT-2 не используются).

Старт: **20M токенов для проверки**. При отсутствии коллапса меняйте бюджет на 500M, затем на 2B. **Не запускайте 2B до проверки 20M.**

Теоретическая база SIGReg: https://github.com/galilai-group/lejepa/blob/main/MINIMAL.md

**Ограничения:** валидационные токены фиксированы по SHA256 ID; при resume не восстанавливается точная позиция стрима, поэтому повторный проход может использовать уже прочитанные train-документы. Retrieval измеряет различимость скрытых токеновых латентов, не доказательство понимания языка. До запуска в Spark код не проверен на GB10.

In [ ]:
import os, math, random, time, json, copy, hashlib
from pathlib import Path
from dataclasses import dataclass, asdict
import numpy as np
import torch
from torch import nn
from torch.nn import functional as F
from torch.utils.data import IterableDataset, DataLoader
from datasets import load_dataset
from transformers import AutoTokenizer
from tqdm.auto import tqdm
import matplotlib.pyplot as plt

DEVICE=torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("PyTorch",torch.__version__,"device",DEVICE)
if DEVICE.type=="cuda":
    print(torch.cuda.get_device_name(0))
    torch.set_float32_matmul_precision("high")

## 1. Конфигурация

Прогон **20M токенов** — диагностика новой функции обучения и контроль того, что SIGReg не даёт латентам схлопнуться. Затем можно увеличить token_budget до 500M и 2B. В отличие от v2, размер фактического набора для SIGReg — 8×128=1024 токеновых эмбеддинга на каждый microbatch; при этом они зависимы.

In [ ]:
@dataclass
class Config:
    seed:int=42
    preset:str="base"  # "large"
    seq_len:int=512
    span_frac:float=0.25
    proj_dim:int=256
    micro_batch:int=8
    grad_accum:int=8
    token_budget:int=20_000_000  # пилот; затем 500_000_000 или 2_000_000_000
    lr:float=2e-4
    weight_decay:float=0.05
    sigreg_weight:float=0.1  # loss=(1-lambda)*MSE + lambda*SIGReg
    sigreg_directions:int=128
    dropout:float=0.0
    max_grad_norm:float=1.0
    eval_every:int=25
    save_every:int=100
    val_batches:int=8
    data_buffer:int=2048
    max_doc_tokens:int=4096
    out_dir:str="runs/text_span_sigreg_v3"
    hf_repo:str="HuggingFaceFW/fineweb-edu"
    hf_config:str="sample-10BT"
    tokenizer:str="gpt2"
cfg=Config()
if cfg.preset=="base":
    depth,width,heads=12,768,12
elif cfg.preset=="large":
    depth,width,heads=16,1024,16
else:
    raise ValueError("preset must be base or large")
assert 0<cfg.span_frac<0.5 and cfg.seq_len>32
assert 0<cfg.sigreg_weight<1, "SIGReg must be enabled"
assert cfg.micro_batch>=2 and cfg.grad_accum>=1
def seed_all(seed):
    random.seed(seed);np.random.seed(seed);torch.manual_seed(seed)
    if torch.cuda.is_available():torch.cuda.manual_seed_all(seed)
seed_all(cfg.seed)
tokens_per_step=cfg.micro_batch*cfg.grad_accum*cfg.seq_len
max_steps=math.ceil(cfg.token_budget/tokens_per_step)
print(f"preset={cfg.preset} depth={depth} d={width} heads={heads}")
print(f"steps={max_steps}, tokens/step={tokens_per_step:,}, token-latents/microbatch ≈ {cfg.micro_batch*round(cfg.seq_len*cfg.span_frac)}")

## 2. FineWeb-Edu через streaming с непересекающимися train/val

Dataset `sample-10BT` скачивается постепенно; исходный корпус целиком в RAM не загружается. Split определяется стабильным SHA256 от `id` документа (или текста при отсутствии `id`): ~1% валидация, остальные — обучение. Два split не пересекаются по этому идентификатору. Это не полная семантическая дедупликация корпуса.

Обучаем на последовательностях в пределах одного документа. Короткие хвосты отбрасываются; поэтому `token_budget` — число токенов, **поданных энкодеру**, а не гарантированное число уникальных прочитанных текстовых токенов. При перезапуске с checkpoint поток исходных данных начинается заново; точное восстановление позиции stream пока не реализовано.

In [ ]:
tokenizer=AutoTokenizer.from_pretrained(cfg.tokenizer)
def is_validation(ex):
    key=str(ex.get("id") or ex.get("text","")[:1000])
    val=int.from_bytes(hashlib.sha256(key.encode("utf-8")).digest()[:8],"big") % 100
    return val == 0
class FineWebBlocks(IterableDataset):
    def __init__(self,part,shuffle=False):
        assert part in ("train","val")
        self.part=part;self.shuffle=shuffle
    def __iter__(self):
        ds=load_dataset(cfg.hf_repo,cfg.hf_config,split="train",streaming=True)
        if self.shuffle:
            ds=ds.shuffle(seed=cfg.seed,buffer_size=cfg.data_buffer)
        for ex in ds:
            if is_validation(ex)!=(self.part=="val"):continue
            text=str(ex.get("text",""))
            if len(text)<30:continue
            ids=tokenizer(text,add_special_tokens=False,truncation=True,max_length=cfg.max_doc_tokens)["input_ids"]
            for i in range(0,len(ids)-cfg.seq_len+1,cfg.seq_len):
                yield torch.tensor(ids[i:i+cfg.seq_len],dtype=torch.long)
train_loader=DataLoader(FineWebBlocks("train",shuffle=True),batch_size=cfg.micro_batch,num_workers=0,pin_memory=DEVICE.type=="cuda")
val_loader=DataLoader(FineWebBlocks("val"),batch_size=cfg.micro_batch,num_workers=0,pin_memory=DEVICE.type=="cuda")
# Зафиксированные данные валидации; повторная оценка не включает новые документы.
val_batches=[]
for i,v in enumerate(val_loader):
    if i>=cfg.val_batches:break
    val_batches.append(v)
assert val_batches,"Validation stream returned no batches."
print("Cached val batches:",len(val_batches))
print("Example:",tokenizer.decode(val_batches[0][0][:64]))

## 3. Маскирование: один скрытый contiguous span

Скрытый участок содержит ~25% токенов. Target encoder получает **только скрытый текст**, context encoder — **только видимый текст**, но оба вызова разделяют одни веса. При обычном обучении начало span случайно; для честной retrieval@1 на валидации **у всех примеров в батче одинаковые скрытые позиции**, чтобы невозможно было угадывать только позицию.

In [ ]:
def split_span(x,randomize=True):
    b,l=x.shape
    span=max(2,round(l*cfg.span_frac))
    if randomize:
        start=torch.randint(0,l-span+1,(b,),device=x.device)
    else:
        # Одинаковое положение для всех примеров, иначе retrieval по позиции нечестен
        start=torch.full((b,), (l-span)//2,device=x.device,dtype=torch.long)
    pos=torch.arange(l,device=x.device)[None,:].expand(b,-1)
    mask=(pos>=start[:,None])&(pos<start[:,None]+span)
    vpos=pos[~mask].reshape(b,l-span)
    tpos=pos[mask].reshape(b,span)
    return x.gather(1,vpos),vpos,x.gather(1,tpos),tpos,start

## 4. Backbone и токеновый Predictor

Общий энкодер 12×768. Target branch теперь выдаёт **128 латентов**, а не один усреднённый вектор. Cross-attention predictor выдаёт отдельный прогноз для каждого скрытого положения. У обоих выходов LayerNorm **без trainable affine-параметров**: это стабилизирует начальный масштаб, но **сам по себе не гарантирует отсутствие коллапса**.

In [ ]:
class Backbone(nn.Module):
    def __init__(self,vocab):
        super().__init__()
        self.word=nn.Embedding(vocab,width)
        self.position=nn.Embedding(cfg.seq_len,width)
        block=nn.TransformerEncoderLayer(width,heads,4*width,dropout=cfg.dropout,activation="gelu",batch_first=True,norm_first=True)
        self.layers=nn.TransformerEncoder(block,depth,enable_nested_tensor=False)
        self.norm=nn.LayerNorm(width)
    def forward(self,ids,positions):
        x=self.word(ids)+self.position(positions)
        return self.norm(self.layers(x))

class TokenSpanHead(nn.Module):
    def __init__(self):
        super().__init__()
        self.target_project=nn.Sequential(
            nn.LayerNorm(width),nn.Linear(width,cfg.proj_dim),
            nn.LayerNorm(cfg.proj_dim,elementwise_affine=False))
        self.query=nn.Embedding(cfg.seq_len,width)
        self.cross=nn.MultiheadAttention(width,heads,batch_first=True)
        self.predictor=nn.Sequential(
            nn.LayerNorm(width),nn.Linear(width,width),nn.GELU(),
            nn.Linear(width,cfg.proj_dim),
            nn.LayerNorm(cfg.proj_dim,elementwise_affine=False))
    def forward(self,context_h,target_h,target_positions):
        queries=self.query(target_positions)
        att,_=self.cross(queries,context_h,context_h,need_weights=False)
        z_pred=self.predictor(queries+att)
        z_target=self.target_project(target_h)
        return z_pred,z_target

encoder=Backbone(len(tokenizer)).to(DEVICE)
head=TokenSpanHead().to(DEVICE)
params=list(encoder.parameters())+list(head.parameters())
print("encoder:",f"{sum(p.numel() for p in encoder.parameters())/1e6:.1f}M params")
print("total trainable:",f"{sum(p.numel() for p in params)/1e6:.1f}M params")
print("One shared encoder. EMA=NONE, stop_gradient=NONE.")

## 5. SIGReg — исправленная формула Sliced Epps–Pulley

В v2 простое среднее комплексных разностей при N=8 давало слабый антиколлапсирующий сигнал. В v3 используем формулу из минимальной реализации **LeJEPA**:

\[
\mathrm{SIGReg}(z)=N\,\mathbb E_{a}\!\int_0^3
2e^{-t^2/2}\Big[(\mathrm{Re}\hat\varphi_a(t)-e^{-t^2/2})^2+
(\mathrm{Im}\hat\varphi_a(t))^2\Big]dt.
\]

Интеграл считается трапецией по 17 узлам, направления случайные единичные. Реализация **не является точной копией библиотеки**, но использует её нормировку/веса; для сравнения с публикацией следует заменить её на установленный официальный модуль.

SIGReg считается по всем B×masked_tokens скрытым позициям **отдельно для predicted и target**. При B=8 и 128 скрытых токенах N=1024, но корреляции уменьшают эффективный N. Эта постановка отличается от LeJEPA на изображениях и требует независимой эмпирической проверки.

Проверка: SIGReg на константных эмбеддингах должен быть **намного выше**, чем на случайном гауссовом наборе. Исходные эмбеддинги дополнительно масштабируются через non-affine LayerNorm.

In [ ]:
class SlicedEppsPulley(nn.Module):
    def __init__(self,knots=17,num_slices=128):
        super().__init__()
        self.num_slices=num_slices
        ts=torch.linspace(0,3,knots,dtype=torch.float32)
        delta=3/(knots-1)
        w=torch.full((knots,),2*delta,dtype=torch.float32)
        w[0]=delta; w[-1]=delta
        gaussian=torch.exp(-0.5*ts.square())
        self.register_buffer("ts",ts)
        self.register_buffer("gaussian",gaussian)
        self.register_buffer("weights",w*gaussian)
    def forward(self,z):
        z=z.reshape(-1,z.shape[-1]).float()
        n,d=z.shape
        assert n>=2, "At least 2 latent samples needed for SIGReg"
        directions=F.normalize(torch.randn(d,self.num_slices,device=z.device),p=2,dim=0,eps=1e-9)
        phase=(z@directions).unsqueeze(-1)*self.ts
        real=phase.cos().mean(dim=0)
        imag=phase.sin().mean(dim=0)
        ep=(real-self.gaussian).square()+imag.square()
        return (ep@self.weights).mean()*n

sigreg=SlicedEppsPulley(num_slices=cfg.sigreg_directions).to(DEVICE)
with torch.no_grad():
    probe_n=1024
    zeros=torch.zeros(probe_n,cfg.proj_dim,device=DEVICE)
    gaussian=torch.randn(probe_n,cfg.proj_dim,device=DEVICE)
    zero_score=float(sigreg(zeros))
    gaussian_score=float(sigreg(gaussian))
print("SIGReg zero vector:",zero_score,"vs Gaussian:",gaussian_score)
assert zero_score > 10*gaussian_score, "SIGReg sanity check failed"

def training_forward(x,randomize=True):
    ctx_ids,cp,tgt_ids,tp,_=split_span(x,randomize)
    context_h=encoder(ctx_ids,cp)
    target_h=encoder(tgt_ids,tp)
    pred,target=head(context_h,target_h,tp)     # [B, masked_tokens, D]
    mse=F.mse_loss(pred.float(),target.float())
    regularizer=0.5*(sigreg(pred)+sigreg(target))
    full=(1-cfg.sigreg_weight)*mse+cfg.sigreg_weight*regularizer
    return full,mse,regularizer,pred,target

opt=torch.optim.AdamW(params,lr=cfg.lr,weight_decay=cfg.weight_decay)

@torch.no_grad()
def evaluate():
    encoder.eval();head.eval()
    rows=[]
    for cpu_x in val_batches:
        x=cpu_x.to(DEVICE)
        with torch.autocast("cuda",dtype=torch.bfloat16,enabled=DEVICE.type=="cuda"):
            loss,lp,lr,pred,target=training_forward(x,False)
        p,t=F.normalize(pred.float(),dim=-1),F.normalize(target.float(),dim=-1)
        # Сравнение при фиксированной позиции: [hidden_position, query_example, candidate_example]
        sims=torch.einsum("btd,ctd->tbc",p,t)
        b=pred.size(0)
        acc=(sims.argmax(-1)==torch.arange(b,device=x.device)[None,:]).float().mean()
        z=target.float().reshape(-1,cfg.proj_dim)
        centered=z-z.mean(0)
        gram=(centered.T@centered)/max(1,len(centered)-1)
        eigen=torch.linalg.eigvalsh(gram).clamp_min(0)
        prob=eigen/(eigen.sum().clamp_min(1e-12))
        rank=torch.exp(-(prob*(prob+1e-12).log()).sum())
        rows.append(dict(pred_loss=float(lp),sigreg=float(lr),at1=float(acc),
            chance_at1=float(1/b),latent_std=float(z.std(0).mean()),rank=float(rank)))
    if not rows: raise RuntimeError("Empty validation")
    return {key:float(np.mean([row[key] for row in rows])) for key in rows[0]}

print("Initial validation:",evaluate())

## 6. Обучение, предупреждение о коллапсе, checkpoint

При первом запуске — 20M токенов. **Проверяйте** \`val_latent_std\`, \`val_rank\`, \`val_sigreg\` и \`val_at1\` относительно \`val_chance_at1\`. Если падение std ниже 0.10 сопровождается большой SIGReg, процесс автоматически остановится (два измерения подряд).

Используется линейный warmup LR + cosine decay; checkpoint сохраняет веса/оптимизатор/число шагов. **Resume не восстанавливает позицию HF streaming**, поэтому после возобновления train документы могут повторяться. Для длинных прогонов потребуется сохранять состояние итератора.

Логирование точной скорости: \`tokens_seen / sec\`, но начальное скачивание HF может снижать средний throughput.

In [ ]:
Path(cfg.out_dir).mkdir(parents=True,exist_ok=True)
checkpoint_path=Path(cfg.out_dir)/"latest.pt"
resume=False
start_step=0
history=[]
if resume:
    blob=torch.load(checkpoint_path,map_location="cpu",weights_only=False)
    for field in ("preset","seq_len","proj_dim","micro_batch","grad_accum","sigreg_weight"):
        assert blob["config"][field]==getattr(cfg,field),f"Config mismatch on {field}"
    encoder.load_state_dict(blob["encoder"])
    head.load_state_dict(blob["head"])
    opt.load_state_dict(blob["optimizer"])
    start_step=blob["step"]
    history=blob.get("history",[])
    print("Resuming from step",start_step)

def save_checkpoint(step):
    tmp=checkpoint_path.with_suffix(".tmp")
    torch.save({"config":asdict(cfg),"step":step,
        "encoder":encoder.state_dict(),"head":head.state_dict(),
        "optimizer":opt.state_dict(),"history":history},tmp)
    tmp.replace(checkpoint_path)

iterator=iter(train_loader)
t0=time.time()
collapse_streak=0
for step in tqdm(range(start_step+1,max_steps+1),desc="Token Text-JEPA + SIGReg v3"):
    encoder.train();head.train()
    opt.zero_grad(set_to_none=True)
    # Warmup + cosine
    warmup=max(10,round(max_steps*0.03))
    if step<=warmup:
        factor=max(0.01,step/warmup)
    else:
        ratio=(step-warmup)/max(1,max_steps-warmup)
        factor=0.1+0.9*0.5*(1+math.cos(math.pi*ratio))
    for group in opt.param_groups: group["lr"]=cfg.lr*factor
    sum_pred=0.;sum_reg=0.
    for micro in range(cfg.grad_accum):
        try: xb=next(iterator)
        except StopIteration:
            iterator=iter(train_loader);xb=next(iterator)
        x=xb.to(DEVICE,non_blocking=True)
        with torch.autocast("cuda",dtype=torch.bfloat16,enabled=DEVICE.type=="cuda"):
            loss,loss_pred,loss_reg,_,_=training_forward(x,True)
        if not torch.isfinite(loss):
            save_checkpoint(step-1)
            raise RuntimeError("Non-finite training loss, checkpoint preserved")
        (loss/cfg.grad_accum).backward()
        sum_pred+=float(loss_pred.detach())/cfg.grad_accum
        sum_reg+=float(loss_reg.detach())/cfg.grad_accum
    gn=float(nn.utils.clip_grad_norm_(params,cfg.max_grad_norm))
    opt.step()
    if step==1 or step%cfg.eval_every==0 or step==max_steps:
        result={"step":step,"tokens_seen":step*tokens_per_step,
            "train_pred":sum_pred,"train_sigreg":sum_reg,
            "grad_norm":gn,"lr":opt.param_groups[0]["lr"],
            "sec":time.time()-t0}
        metrics=evaluate()
        result.update({"val_"+k:v for k,v in metrics.items()})
        history.append(result)
        print(result)
        collapse_risk=(metrics["latent_std"]<0.1 and metrics["sigreg"]>20)
        collapse_streak=collapse_streak+1 if collapse_risk else 0
        if collapse_streak>=2:
            save_checkpoint(step)
            print("COLLAPSE WARNING: latents near-constant. Stopped early.")
            break
    if step%cfg.save_every==0 or step==max_steps:
        save_checkpoint(step)
if history:
    save_checkpoint(history[-1]["step"])
print("Finished step:",history[-1]["step"] if history else start_step)
print("Saved local checkpoint:",checkpoint_path)

## 7. Отчёт и сравнение с v2

Выгрузка не включает исходные тексты, веса или GitHub-токены. Для v3 определение \`val_at1\` изменилось: теперь кандидаты взяты **из одного скрытого положения** в батче, так что нельзя просто угадывать индекс токена. Сравнение абсолютных retrieval@1 между v2 и v3 неэквивалентно.

**Критерий успеха пилота:** std не падает почти к нулю; SIGReg не растёт до сотен; retrieval@1 выше chance на фиксированных позициях. Важно проверить downstream linear probes, а не только самосогласованность.

In [ ]:
def export_metrics():
    out=Path(cfg.out_dir)
    out.mkdir(parents=True,exist_ok=True)
    (out/"config.json").write_text(json.dumps(asdict(cfg),indent=2,ensure_ascii=False))
    (out/"metrics.json").write_text(json.dumps({
        "method":"v3 token-wise span JEPA + Epps-Pulley SIGReg (no EMA)",
        "schema_version":3,
        "sigreg_n_per_microbatch":cfg.micro_batch*round(cfg.seq_len*cfg.span_frac),
        "note":"SIGReg token embeddings are correlated; stream cursor not restored upon resume",
        "tokens_seen":history[-1]["tokens_seen"] if history else 0,
        "history":history,
        "validation_holdout_hash":True,
        "metrics_note":"val_at1 is same-hidden-position retrieval; chance=1/micro_batch"
    },indent=2,ensure_ascii=False))
    fig,axs=plt.subplots(1,4,figsize=(17,4))
    for ax,k in zip(axs,["val_pred_loss","val_sigreg","val_latent_std","val_at1"]):
        ax.plot([v["tokens_seen"]/1e6 for v in history],[v[k] for v in history],marker=".")
        ax.set_title(k);ax.set_xlabel("tokens (M)");ax.grid(alpha=.3)
        if k=="val_at1" and history:
            ax.axhline(history[-1]["val_chance_at1"],linestyle="--",label="random")
            ax.legend()
    fig.tight_layout()
    fig.savefig(out/"training.png",dpi=140)
    plt.show();plt.close(fig)
    print("Exported:",out/"config.json",out/"metrics.json",out/"training.png")
    return out
export_dir=export_metrics()

## 8. Публикация в GitHub (опционально)

Публикуются **только три отчётных файла** в отдельном каталоге \`results/text_span_sigreg_v3\`: конфигурация, метрики и график. Не публикуются модельные веса, токены GitHub, датасет или рабочая копия ноутбука. Git должен быть настроен на вашей машине. Ветку менять не нужно.

In [ ]:
import subprocess
from datetime import datetime,timezone
EXPECTED_BRANCH="experiment/text-i-jepa-notebook-20261113"
def run_git(*args,root=None,timeout=90):
    p=subprocess.run(["git",*args],cwd=root,stdout=subprocess.PIPE,
                     stderr=subprocess.PIPE,text=True,timeout=timeout)
    if p.returncode:
        raise RuntimeError("git "+ " ".join(args)+" failed: "+p.stderr[-1000:])
    return p.stdout.strip()
def publish_v3_report():
    if not history: raise RuntimeError("Train first")
    root=Path(run_git("rev-parse","--show-toplevel")).resolve()
    if run_git("branch","--show-current",root=root)!=EXPECTED_BRANCH:
        raise RuntimeError("Open the experimental branch before publishing.")
    origin=run_git("remote","get-url","origin",root=root)
    if origin.rstrip("/").removesuffix(".git") not in (
        "https://github.com/crystalcali/salm",
        "git@github.com:crystalcali/salm",
        "ssh://git@github.com/crystalcali/salm"):
        raise RuntimeError(f"Unexpected origin: {origin}")
    if run_git("diff","--cached","--name-only",root=root):
        raise RuntimeError("Git index already has staged files; resolve before publishing")
    source=export_metrics()
    timestamp=datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")
    rel=Path("results")/"text_span_sigreg_v3"/(timestamp+"_"+cfg.preset+"_"+str(cfg.seed))
    dest=root/rel;dest.mkdir(parents=True,exist_ok=False)
    for name in ("config.json","metrics.json","training.png"):
        (dest/name).write_bytes((source/name).read_bytes())
    paths=[str(rel/name) for name in ("config.json","metrics.json","training.png")]
    run_git("add","--",*paths,root=root)
    if set(run_git("diff","--cached","--name-only",root=root).splitlines())!=set(paths):
        run_git("reset","--",*paths,root=root)
        raise RuntimeError("Unexpected paths staged")
    run_git("commit","-m",f"results: text SIGReg v3 {timestamp}",root=root)
    run_git("push","origin",f"HEAD:refs/heads/{EXPECTED_BRANCH}",root=root,timeout=120)
    url="https://github.com/crystalcali/salm/tree/"+EXPECTED_BRANCH+"/"+rel.as_posix()
    print("Published:",url)
    return url

# Вызовите отдельно после обучения:
# report_url=publish_v3_report()